In [2]:
import time
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer,HashingVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# reading csv files
dataset=pd.read_csv('Books1.csv', low_memory=False)
ratings=pd.read_csv('Ratings1.csv')
users=pd.read_csv('Users1.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Books1.csv'

In [3]:
# dropping irrelevant columns
dataset.drop(['Image-URL-S','Image-URL-M','Image-URL-L'],axis=1,inplace=True)
# converting data to integer data type
ratings['User-ID']=ratings['User-ID'].astype('int')
users['User-ID']=users['User-ID'].astype('int')

In [4]:
# merging with users and ratings dataset to get one combined dataset
dataset=dataset.merge(ratings,on='ISBN')
dataset=dataset.merge(users,on='User-ID')
dataset.rename(columns={'Age':'User-Age','Location':'User-Location'},inplace=True)

In [5]:
print(dataset.shape)
dataset.head()

(1031136, 9)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,User-ID,Book-Rating,User-Location,User-Age
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,2,0,"stockton, california, usa",18.0
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,8,5,"timmins, ontario, canada",NaN
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,8,0,"timmins, ontario, canada",NaN
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,8,0,"timmins, ontario, canada",NaN
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,8,0,"timmins, ontario, canada",NaN


In [6]:
# seeing how many null values are present
dataset.isnull().sum()

ISBN                        0
Book-Title                  0
Book-Author                 1
Year-Of-Publication         0
Publisher                   2
User-ID                     0
Book-Rating                 0
User-Location               0
User-Age               277835
dtype: int64

In [7]:
# dropping all rows with null values in them
dataset.dropna(inplace=True)

In [8]:
# number of rows to be sampled from dataset
# more number of rows causes memory errors
max_num=100000

In [9]:
# Removing rows with duplicate book title
dataset=dataset.loc[~dataset['Book-Title'].duplicated(),:]
# sampling max_num rows from the dataset
dataset=dataset.sample(n=max_num,random_state=1)
dataset.reset_index(drop=True,inplace=True)
dataset.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,User-ID,Book-Rating,User-Location,User-Age
0,0075536684,Paradise Lost,John Milton,1969,McGraw-Hill Humanities/Social Sciences/Languages,177458,0,"ottawa, ontario, canada",29.0
1,0025005731,The Great Ideas: A Lexicon of Western Thought,Mortimer J. Adler,1992,MacMillan Publishing Company.,84716,0,"auckland, auckland,",44.0
2,0671472127,HT DEVELOP SLF CON,Dale Carneigie,1983,Pocket,125692,0,"elmira, new york, usa",28.0
3,2258061474,Parlez-vous le Jean-Claude ?,Dominique Duforest,2003,Hors Collection,53628,10,"brest, bretagne, france",22.0
4,0842355065,"Left Behind Graphic Novel (Book 1, Volume 5)",John S. Layman,2002,Tyndale House Publishers,233917,0,"madera, california, usa",38.0


In [10]:
# changing to lower case, removing whitespace and converting to list data type
dataset['Book-Author']=dataset['Book-Author'].apply(lambda x: [str.lower(x.replace(' ',''))])
dataset['Publisher']=dataset['Publisher'].apply(lambda x: [str.lower(x.replace(' ',''))])
dataset['Year-Of-Publication']=dataset['Year-Of-Publication'].apply(lambda x: [str(x)])

In [11]:
# creating metadata_dump column
# Book-Author, Publisher and year of publication are considered
dataset['metadata_dump'] =dataset['Book-Author'] + dataset['Publisher'] + dataset['Year-Of-Publication']
dataset['metadata_dump'] = dataset['metadata_dump'].apply(lambda x: ' '.join(map(str,x)))


In [15]:
import numpy as np
t0 = time.time()
# Initializing count matrix
count = CountVectorizer(analyzer='word',ngram_range=(1, 2),min_df=0, stop_words='english')
# count=HashingVectorizer(analyzer='word',ngram_range=(1, 2),n_features=2**18, stop_words='english')
count_matrix = count.fit_transform(dataset['metadata_dump'])


# calculating cosine similarity matrix
count_matrix = count_matrix.astype(np.float32)
cosine_sim = cosine_similarity(count_matrix, count_matrix)
duration = time.time() - t0
print(f"done in {duration:.3f} s")

done in 204.400 s


In [16]:
cosine_sim.shape

(100000, 100000)

In [17]:
# dataset['metadata_dump'][0]

In [18]:
# print(count.vocabulary_)

In [19]:
# import ast
# dicti=ast.literal_eval(str(count.vocabulary_))
# keys = [k for k, v in dicti.items() if v == 51]
# print(keys)

In [20]:
# print(count_matrix[0])

In [21]:
# Extracting Book titles along with their indexes
titles = dataset['Book-Title']
indices = pd.Series(dataset.index, index=dataset['Book-Title'])

In [22]:
# returns dataframe of top 10 books with highest cosine similarity to input book along with common features with input book
recc_idx=[]
def get_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(list(cosine_sim[idx])))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # we take from index 1 onwards so that a book does not show up as a recommendation for itself
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    
    
    recc=titles.iloc[movie_indices]
    recc=recc.to_frame()
    recc['Common Features']=''
    query_author=dataset[dataset['Book-Title']==title]['Book-Author'].values
    query_yop=dataset[dataset['Book-Title']==title]['Year-Of-Publication'].values
    query_publisher=dataset[dataset['Book-Title']==title]['Publisher'].values
    for i,row in recc.iterrows():
        recc_idx.append(i)
        common=[]
        recc_title=row['Book-Title']
        recc_author=dataset[dataset['Book-Title']==recc_title]['Book-Author'].values
        recc_yop=dataset[dataset['Book-Title']==recc_title]['Year-Of-Publication'].values
        recc_publisher=dataset[dataset['Book-Title']==recc_title]['Publisher'].values
        if recc_author==query_author:
            common.append('Book-Author')
        if recc_yop==query_yop:
            common.append('Year-Of-Publication')
        if recc_publisher==query_publisher:
            common.append('Publisher')
        recc.loc[i,'Common Features']=', '.join(common)
    return recc

In [23]:
# To see which books we can make a prediction for since sampling was done
indices

Book-Title
Paradise Lost                                                 0
The Great Ideas: A Lexicon of Western Thought                 1
HT DEVELOP SLF CON                                            2
Parlez-vous le Jean-Claude ?                                  3
Left Behind Graphic Novel (Book 1, Volume 5)                  4
                                                          ...  
The Harper Collins World Reader: Single Volume Edition    99995
Savage Pagan                                              99996
Hummingbird                                               99997
Annie's Wild Ride                                         99998
The Blue Between the Clouds                               99999
Length: 100000, dtype: int64

In [24]:
get_recommendations('Hummingbird').head(10)

,Book-Title,Common Features
99997,Hummingbird,"Book-Author, Year-Of-Publication, Publisher"
1180,"True Romance (Silver Creek Riders, No 2)","Year-Of-Publication, Publisher"
1579,Prime Witness,"Year-Of-Publication, Publisher"
3201,Private Scandals,"Year-Of-Publication, Publisher"
10018,November of the Heart,"Book-Author, Publisher"
10021,The Duke Finds Love (Camfield No 129),"Year-Of-Publication, Publisher"
13191,Death in Ecstasy,"Year-Of-Publication, Publisher"
13838,Stolen Kisses,"Year-Of-Publication, Publisher"
25041,The Captains: Brotherhood of War (Brotherhood ...,"Year-Of-Publication, Publisher"
26915,Skull and Dog Bones (Dog Lover's Mystery),"Year-Of-Publication, Publisher"


In [25]:
# To check if the common features column is correct
dataset.loc[recc_idx]

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,User-ID,Book-Rating,User-Location,User-Age,metadata_dump
99997,051509160X,Hummingbird,[lavyrlespencer],[1994],[jovebooks],52614,0,"toccoa, ga., usa",33.0,lavyrlespencer jovebooks 1994
1180,0515114898,"True Romance (Silver Creek Riders, No 2)",[bethkincaid],[1994],[jovebooks],65258,0,"saginaw, michigan, usa",42.0,bethkincaid jovebooks 1994
1579,051511264X,Prime Witness,[stevenpaulmartini],[1994],[jovebooks],36606,0,"san marcos, california, usa",39.0,stevenpaulmartini jovebooks 1994
3201,0515114006,Private Scandals,[noraroberts],[1994],[jovebooks],211919,0,"jacksonville, florida, usa",38.0,noraroberts jovebooks 1994
10018,051511331X,November of the Heart,[lavyrlespencer],[1995],[jovebooks],177090,0,"ashland, missouri, usa",37.0,lavyrlespencer jovebooks 1995
10021,0515113786,The Duke Finds Love (Camfield No 129),[barbaracartland],[1994],[jovebooks],274061,0,"gahanna/columbus, ohio, usa",26.0,barbaracartland jovebooks 1994
13191,0515085928,Death in Ecstasy,[ngaiomarsh],[1994],[jovebooks],240144,8,"muskego, wisconsin, usa",34.0,ngaiomarsh jovebooks 1994
13838,0515114901,Stolen Kisses,[karenlockwood],[1994],[jovebooks],26544,0,"woodbridge, virginia, usa",37.0,karenlockwood jovebooks 1994
25041,0515091383,The Captains: Brotherhood of War (Brotherhood ...,[w.e.b.griffin],[1994],[jovebooks],39646,0,"fairless hills, pennsylvania, usa",35.0,w.e.b.griffin jovebooks 1994
26915,0515112798,Skull and Dog Bones (Dog Lover's Mystery),[melissacleary],[1994],[jovebooks],207782,0,"midland, texas, usa",28.0,melissacleary jovebooks 1994


In [26]:
# Finding out why 'Hummingbird' shows up as a recommendation for itself
# a book should have the maximum cosine similarity with itself, therefore when outputting reccomendations we skip the first book in descending cosine similarity scores.
# However here there is another book that has maximum cosine similarity score with Hummingbird as well
maxx=max(cosine_sim[99997])
for i,num in enumerate(cosine_sim[99997]):
    if num==maxx:
        print(f'{i}   {num}')

7950   0.9999999403953552
99997   0.9999999403953552


In [27]:
# finding which book-title it is that has the same maximum cosine similarity
indices.index[7950]

'The Endearment'

In [28]:
# seeing the metadata_dump for the book
dataset[dataset['Book-Title']=='The Endearment']

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,User-ID,Book-Rating,User-Location,User-Age,metadata_dump
7950,0515103969,The Endearment,[lavyrlespencer],[1994],[jovebooks],207148,0,"san diego, california, usa",42.0,lavyrlespencer jovebooks 1994


In [29]:
# seeing the metadata_dump for the input book, the medata_dumps are the same
dataset[dataset['Book-Title']=='Hummingbird']

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,User-ID,Book-Rating,User-Location,User-Age,metadata_dump
99997,051509160X,Hummingbird,[lavyrlespencer],[1994],[jovebooks],52614,0,"toccoa, ga., usa",33.0,lavyrlespencer jovebooks 1994
